# 04. So sánh Mô hình, Trực quan hóa Ma trận Chuyển CRF & Phân tích Lỗi

Notebook của **Đạt**: tổng hợp kết quả từ `outputs/metrics/*.json`, vẽ biểu đồ F1 cho từng loại trong 10 thực thể, trực quan hóa ma trận `transitions` của tầng `LinearChainCRF`, và gọi `categorize_errors` để trích xuất bảng lỗi phục vụ báo cáo ACL.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.download import load_phoner_split
from src.data.labels import ENTITY_TYPES, LABELS, decode_tags
from src.evaluation.error_analysis import categorize_errors
from src.training.trainer import build_model_and_dataloaders

METRICS_DIR = ROOT / "outputs" / "metrics"
FIG_DIR = ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

experiments = {}
for p in sorted(METRICS_DIR.glob("*.json")):
    data = json.loads(p.read_text(encoding="utf-8"))
    experiments[data["experiment_name"]] = data
    t = data["test"]
    print(
        f"{data['experiment_name']:28s} | P={t['micro_precision']:.4f} | "
        f"R={t['micro_recall']:.4f} | micro-F1={t['micro_f1']:.4f} | "
        f"macro-F1={t['macro_f1']:.4f} | BIO_viol={t['bio_violations']}"
    )

In [ ]:
if experiments:
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(ENTITY_TYPES))
    n_models = len(experiments)
    width = 0.8 / max(1, n_models)

    for idx, (name, data) in enumerate(experiments.items()):
        f1s = [data["test"]["per_entity"][e]["f1"] for e in ENTITY_TYPES]
        ax.bar(x + (idx - (n_models - 1) / 2) * width, f1s, width=width, label=name)

    ax.set_xticks(x)
    ax.set_xticklabels(ENTITY_TYPES, rotation=35, ha="right")
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Entity F1 (Strict IOB2)")
    ax.set_title("Per-Entity F1 Comparison on PhoNER_COVID19 Test Set")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "per_entity_f1_comparison.png", dpi=200)
    plt.show()

In [ ]:
ckpt_path = ROOT / "checkpoints" / "bilstm_crf" / "best_model.pt"
if ckpt_path.exists():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    test_records = load_phoner_split(ROOT / cfg["data"]["data_dir"], level=cfg["data"]["level"], split="test")
    model, loaders, _ = build_model_and_dataloaders(cfg, {"test": test_records}, vocab_state=ckpt.get("vocab_state"))
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device).eval()

    if getattr(model, "crf", None) is not None:
        trans = model.crf.transitions.detach().cpu().numpy()
        plt.figure(figsize=(12, 10))
        sns.heatmap(trans, xticklabels=LABELS, yticklabels=LABELS, cmap="coolwarm", center=0.0)
        plt.title("Learned CRF Transition Matrix (from row i to col j)")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "crf_transitions_heatmap.png", dpi=200)
        plt.show()

    words_all, y_true, y_pred = [], [], []
    with torch.no_grad():
        for batch in loaders["test"]:
            out = model(
                word_ids=batch["word_ids"].to(device),
                char_ids=batch["char_ids"].to(device),
                mask=batch["mask"].to(device),
            )
            words_all.extend(batch["words"])
            y_true.extend(batch["gold_tags"])
            y_pred.extend([decode_tags(seq) for seq in out["predictions"]])

    report = categorize_errors(words_all, y_true, y_pred)
    print("Error counts:", report["counts"])
    for cat, items in report["examples"].items():
        print(f"\n--- {cat} (showing up to 3) ---")
        for ex in items[:3]:
            print(ex)
else:
    print(f"Checkpoint not found at {ckpt_path}; run training first.")